In [1]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have
'''

import sys
from datetime import datetime

#printing the python version, executable path and current date and time to verify the environment setup.
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running Date and time: 2026-11-02 13:04:04


In [2]:
''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# test_file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# test_file_path = r"\sample_data\test-product-review.parquet"
# train_file_path = r"C:\Users\veera\AI\SampleFile\train-product-review.parquet"
# train_file_path = r"\sample_data\train-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive') #Mounting the Google Drive to access the file stored in it.
# While mounting the drive there might me popups asking to provide permission.
print("Drive mounted:", os.path.ismount('/content/drive'))

train_file_path = '/content/drive/MyDrive/AI/SampleFile/train-product-review.parquet'
print("File exists:"+train_file_path+" -", os.path.isfile(train_file_path)) #checking if the file exists in the specified path.

test_file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+test_file_path+" -", os.path.isfile(test_file_path)) #checking if the file exists in the specified path.



Mounted at /content/drive
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/train-product-review.parquet - True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True


In [3]:
# Read the Parquet file 
# with specific columns and filter for English language reviews.
# For Classical ML we are filtering only En. Tried Translation but it is not giving good results. So, we are using only English reviews for Classical ML.
# For better understanding I have filtered the data for few columns only. You can remove the filter if you want to read all the data.

from time import time

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

train_table = pq.read_table(train_file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])
test_table = pq.read_table(test_file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])

df_train = train_table.to_pandas()
df_test = test_table.to_pandas()

# Print the first few rows of the DataFrame to see how the data looks like.
df_train.info()
print(df_train.head(4))
#printing the count of each star rating in the dataset to understand the distribution of ratings.
print("Count of each star rating:"+str(df_train["stars"].value_counts().to_dict()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6736 entries, 0 to 6735
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    6736 non-null   object
 1   review_body  6736 non-null   object
 2   language     6736 non-null   object
 3   stars        6736 non-null   int32 
dtypes: int32(1), object(3)
memory usage: 184.3+ KB
    review_id                                        review_body language  \
0  en_0364619     Great smell and tasting tea. Love the box too.       en   
1  en_0728455  Love, love, love the designs! Very pretty! But...       en   
2  en_0517240  It's a really good battery. Works well and you...       en   
3  en_0301228             Honestly Didn't hold anything in place       en   

   stars  
0      5  
1      3  
2      3  
3      3  
Count of each star rating:{3: 1406, 5: 1353, 1: 1337, 2: 1320, 4: 1320}


In [4]:
'''DATA PRE-PROCESSING
"tokens", "lemmas", "stems"
For cleaning the data we will use all the 3 above techniques. 
While doing the execution fount that NOT and NEGATION words are important for sentiment analysis. So we will keep them in the stopwords list.
Along with that we will also remove the URLs, special characters, and extra spaces.
After applying the above techniques we will create a new column called "review_body_processed" by combining all the words(tokens) which will be used for vectorization and model training.
'''
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download these resources once
nltk.download("punkt")
nltk.download("punkt_tab")  #
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
# Keep negation words because they can change review sentiment
stop_words -= { "no", "not", "never", "none", "neither", "nor", "noone", "nobody", "nothing", "nowhere", "hardly", "scarcely", "barely", "seldom", "rarely", "don't", "doesn't", "didn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't", "won't", "wouldn't", "shouldn't", "can't", "cannot", "couldn't", }

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
# Remove URLs, special characters, and extra spaces
def preprocess_review(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]

    # Apply lemmatization, then stemming
    lemmas = [
        lemmatizer.lemmatize(
            lemmatizer.lemmatize(token, pos="v"),
            pos="n"
        )
        for token in tokens
    ]
    stems = [stemmer.stem(token) for token in lemmas]

    return tokens, lemmas, stems

df_train[["tokens", "lemmas", "stems"]] = df_train["review_body"].apply(
    lambda text: __import__("pandas").Series(preprocess_review(text))
)

# New Text column so we can keep the original review_body for reference and use the processed one for model training.
df_train["review_body_processed"] = df_train["stems"].apply(" ".join)

#Printing the first few rows of the DataFrame to see how the data looks like after preprocessing.
print(df_train[["review_body", "review_body_processed"]].head())
df_train.info() #Printing info to see how many column we have

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


                                         review_body  \
0     Great smell and tasting tea. Love the box too.   
1  Love, love, love the designs! Very pretty! But...   
2  It's a really good battery. Works well and you...   
3             Honestly Didn't hold anything in place   
4       Great Product! Arrived on time as specified.   

                               review_body_processed  
0                      great smell tast tea love box  
1      love love love design pretti wear design eras  
2  's realli good batteri work well charg lot tim...  
3                      honestli n't hold anyth place  
4                   great product arriv time specifi  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6736 entries, 0 to 6735
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   review_id              6736 non-null   object
 1   review_body            6736 non-null   object
 2   language          

In [5]:
'''DATA CLEANING AND LABEL PREPARATION
Here we will convert the "stars" column to numeric and create a new "sentiment" column based on the star ratings.
We wil keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
We might also need to drop and empty or null values in the review_body"stars" column, as they cannot be used for sentiment analysis. Since Data is clean commenting it out. If you want to use it, you can uncomment the code below.
'''

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Drop rows with missing values in "review_body" and "stars" columns, and create a copy of the DataFrame for further processing.
# df = df[["review_body", "stars"]].dropna().copy()

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df_train["stars"] = pd.to_numeric(df_train["stars"], errors="coerce")


# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df_train["sentiment"] = df_train["stars"].apply(create_sentiment_label)
df_test["sentiment"] = df_test["stars"].apply(create_sentiment_label)

# Printing the first few rows to check and total Counts
print(df_train[["stars", "sentiment"]].head(6))
print(df_train["stars"].value_counts())
print(df_train["sentiment"].value_counts())


   stars sentiment
0      5  positive
1      3   neutral
2      3   neutral
3      3   neutral
4      5  positive
5      2  negative
stars
3    1406
5    1353
1    1337
2    1320
4    1320
Name: count, dtype: int64
sentiment
positive    2673
negative    2657
neutral     1406
Name: count, dtype: int64


In [6]:
#~~~~~~~~ Train/Test Split. ~~~~~~~~~~~~~~
# we use 70% of the data for training and 30% for testing, 
# stratified across the 3 sentiment classes to ensure that each  is represented in both the training and testing .
'''X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["review_body_processed"],
    df["sentiment"],
    test_size=0.3,
    random_state=42,
   stratify =df["sentiment"],
)'''

# Extract features and labels directly from your separate training set
X_train_text = df_train["review_body_processed"]
y_train = df_train["sentiment"]

# Extract features and labels directly from your separate testing set
X_test_text = df_test["review_body"]
y_test = df_test["sentiment"]

# -----Feature Extraction (TF-IDF with unigrams and bigrams)----------------
# We use 2 word combinations (bigrams) along with single words (unigrams) to capture more context in the text data. 
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
# X_test = vectorizer.transform(X_test_text)
X_test = (X_test_text)
# Printing results for validation
print ("Train shape (Rows + columns of words):", X_train.shape)
print ("Test shape (Rows + columns of words):", X_test.shape)
print("Class distribution in train:", y_train.value_counts().to_dict())

Train shape (Rows + columns of words): (6736, 86624)
Test shape (Rows + columns of words): (460,)
Class distribution in train: {'positive': 2673, 'negative': 2657, 'neutral': 1406}


In [7]:
''' Logistic Regression Model Training and Evaluation
We will train a Logistic Regression model on the training data and evaluate its performance on the test data
'''
from sklearn.metrics import accuracy_score, classification_report

# Initialize and Train Logistic Regression
# Make sure class_weight is set to "balanced" to handle neutral class imbalance. This will help the model to give equal importance to all classes during training. 
# If not this will lead to bias towards the majority class and poor performance on the neutral sentiment.
log_reg = LogisticRegression(class_weight="balanced")
log_reg.fit(X_train, y_train)

# Predict and Evaluate
y_pred = log_reg.predict(X_test)
print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Before doing this we will create array to store the results of the models for comparison later. We will create a list of dictionaries to store the model name, accuracy, precision, recall, and F1-score for each model.
results = []
model_names = []
def get_metrics(y_true, y_pred):
  acc = accuracy_score(y_true, y_pred)
  prec, rec, f1, _ = precision_recall_fscore_support(
      y_true, y_pred, average="weighted", zero_division=0
  )
  return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}

results.append(get_metrics(y_test, y_pred))
model_names.append("LogisticRegression ")


ValueError: could not convert string to float: "Well I haven't stunned anyone...yet. So I can only rate what I know. I bought this because it was small and I could attach it to my belt. The belt loop strap on the case broke in the first 4 hours I had it. So now the convienence of being able to carry it on my side is lost. Now I'm going to have to invest in a case to carry it in."

In [26]:
# Now we will do hyperparameter tuning for the Logistic Regression model using GridSearchCV to find the best C value and class_weight parameter.
# Checking the C  value of the model to see how well it is performing. The C value is a regularization parameter that controls the trade-off between achieving a low error on the training data and minimizing the model complexity. 
# A smaller C value means stronger regularization, which can help prevent overfitting but may lead to underfitting if set too high. 
# A larger C value means weaker regularization, which can lead to overfitting if set too low. 
# The optimal C value depends on the specific dataset and problem being solved, and it is often determined through cross-validation or hyperparameter tuning.

from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search over 
param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "class_weight": [None, "balanced"],
}

# Initialize the base Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring="f1_weighted",  # Optimize for weighted F1-score to handle 3 classes
    n_jobs=-1,  # Use all available CPU cores
)

# Fit the grid search on the training data
grid_search.fit(X_train, y_train)

# Print the best parameters found
print("Best Parameters Found:")
print(grid_search.best_params_)
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}\n")

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- Logistic Regression - Hyperparameter Tuning---")
print(classification_report(y_test, y_pred, zero_division=0))

# append the results of the hyperparameter tuned model to the results list for comparison with other models.
results.append(get_metrics(y_test, y_pred))
model_names.append("LogisticRegression Hyperparameter Tuned")


In [27]:
'''Lets do 2 more model fit and evaluation in CLassification task. 
We will use Multinomial Naive Bayes and Linear SVC models for this purpose.
'''

# ~~~~~~"Linear SVM (LinearSVC)" Fit , predict and evaluate~~~~~~~~~
Linear_SVC = LinearSVC(random_state=42)
Linear_SVC.fit(X_train, y_train)
y_pred = Linear_SVC.predict(X_test)
# append the results of the LinearSVC model to the results list for comparison with other models.
results.append(get_metrics(y_test, y_pred))
model_names.append("LinearSVC")

#~~~~~~~~~~~ Multinomial Naive Bayes~~~~~~~
MNB=MultinomialNB()

MNB.fit(X_train, y_train)
y_pred = MNB.predict(X_test)
# append the results of the MultinomialNB model to the results list for comparison with other models.
results.append(get_metrics(y_test, y_pred))
model_names.append("MultinomialNB")

# Create and display the unified comparison DataFrame
comparison_df = pd.DataFrame(results, index=model_names)

# Printing the side-by-side comparison of the model performance metrics 
print("=" * 65)
print("             SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           ")
print("=" * 65)
print(comparison_df.round(4))
print("=" * 65)


In [28]:
'''Deep Learning Model Training and Evaluation 
we will use the Hugging Face Transformers library to fine-tune a pre-trained model for sentiment classification.
using distilbert-base-multilingual-cased so we can handle multiple languages.
Before going to deep learning model, we will do some data preparation and preprocessing as we need import all Languages data for deep learning model.

'''
from datasets import Dataset
!pip install -q optuna
import optuna  # Required for hyperparameter search
import logging
from transformers import logging as transformers_logging

# Suppress Hugging Face info/warning logs . There are many info/warning logs generated
transformers_logging.set_verbosity_error()
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

from transformers import (
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)


# table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
#                       filters=[[('language', '==', 'en')]])
table_train = pq.read_table(train_file_path, columns=['review_id','review_body', 'language','stars'],
                      )
table_test = pq.read_table(test_file_path, columns=['review_id','review_body', 'language','stars'],)
    
# #Print the first few rows of the DataFrame to see how the data looks like.
df_deep_train = table_train.to_pandas()
df_deep_test = table_test.to_pandas()

# Drop rows where either 'review_body' or 'stars' is missing
initial_count = len(df_deep_train)
# df_deep_train = df_deep_train.dropna(subset=["review_body", "stars"])
final_count = len(df_deep_train)
dropped_count = initial_count - final_count
print(f"Count of rows dropped: {dropped_count}")#prnting to see how many rows were dropped due to missing values in 'review_body' or 'stars'.

print("Count of each star rating:" + str(df_deep_train["stars"].value_counts().to_dict()))

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df_deep_train["stars"] = pd.to_numeric(df_deep_train["stars"], errors="coerce")
df_deep_test["stars"] = pd.to_numeric(df_deep_test["stars"], errors="coerce")

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df_deep_train["sentiment"] = df_deep_train["stars"].apply(create_sentiment_label)
df_deep_test["sentiment"] = df_deep_test["stars"].apply(create_sentiment_label)


# Map string labels to integers (Required for Transformer models)
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df_deep_train["label"] = df_deep_train["sentiment"].map(label_mapping)
df_deep_test["label"] = df_deep_test["sentiment"].map(label_mapping)

# Train/Test Split (Stratified)
# train_df, test_df = train_test_split(
#     df_deep, test_size=0.3, random_state=42, stratify=df_deep["label"]
# )
train_df = df_deep_train
test_df = df_deep_test

# Convert Pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

In [29]:
# now we have the train and test datasets we move to the next step of tokenization and model training.
# Tokenization using Multilingual DistilBERT Tokenizer
import numpy as np
model_name = "distilbert-base-multilingual-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)


def tokenize_function(examples):
  return tokenizer(examples["review_body"], truncation=True, max_length=128)

# Split the datasets into training and test sets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
# printing the number of training and test samples and a sample of tokenized features to verify .
print(f"Number of training samples: {len(tokenized_train)}")
print(f"Number of test samples: {len(tokenized_test)}")
print("Sample tokenized features:", tokenized_train[0]["input_ids"])


# Model Initialization Function (Required for Optuna hyperparameter search)
def model_init(trial):
  return DistilBertForSequenceClassification.from_pretrained(
      model_name, num_labels=3, id2label=inv_label_mapping, label2id=label_mapping
  )


# Define Evaluation Metric Function
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  preds = np.argmax(predictions, axis=1)
  accuracy = np.mean(preds == labels)
  return {"accuracy": accuracy}


# Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./distilbert_multilingual_results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True,#added for faster training on GPU as it took 10 min in CPU
    dataloader_num_workers=2,#used for faster data loading as it allowsmultiple worker processes to load data in parallel.
    per_device_eval_batch_size=32, #to allows multiple samples to be processed in parallel 
)

# Initialize Trainer with model_init
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


# Define Hyperparameter Search Space for Optuna
def hp_space(trial):
  return {
      "learning_rate": trial.suggest_float(
          "learning_rate", 1e-5, 5e-5, log=True
      ),
      "per_device_train_batch_size": trial.suggest_categorical(
          "per_device_train_batch_size", [4, 8, 16]
      ),
      "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
      "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
  }


# Running Hyperparameter Tuning
print("Starting Optuna Hyperparameter Search...")
best_run = trainer.hyperparameter_search(
    direction="maximize", backend="optuna", hp_space=hp_space, n_trials=5
)

print("\nBest Hyperparameters Found:")
print(best_run.hyperparameters)

# Apply Best Hyperparameters & Finalize Training
print("\nApplying Best Hyperparameters for Final Training...")
for key, value in best_run.hyperparameters.items():
  setattr(trainer.args, key, value)

trainer.train()

# Evaluate on the Test Set
from sklearn.metrics import precision_recall_fscore_support

print("\nEvaluating Best Model on Test Data:")
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

# Extract predictions for Multilingual DistilBERT to compute metrics
preds_output = trainer.predict(tokenized_test)
preds = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids




In [30]:
accuracy = eval_results.get("eval_accuracy", 0.0)

precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, preds, average="weighted"
)
accuracy = eval_results.get("eval_accuracy", 0.0)

# Append results using a distinct name to reflect the multilingual model
results.append(
    {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1}
)
model_names.append("DistilBERT-Multilingual")

# Create and display the unified comparison DataFrame
comparison_df = pd.DataFrame(results, index=model_names)

print("=" * 65)
print("            SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           ")
print("=" * 65)
print(comparison_df.round(4))
print("=" * 65)

# Compute accuracy and sample size for each language group
language_performance = []

eval_df = test_df.reset_index(drop=True).copy()
eval_df["predicted_label"] = preds

for lang, group in eval_df.groupby("language"):
  acc = accuracy_score(group["label"], group["predicted_label"])
  sample_count = len(group)
  language_performance.append(
      {"Language": lang, "Accuracy": acc, "Samples": sample_count}
  )

lang_perf_df = pd.DataFrame(language_performance)
# Display the performance breakdown table
print("=" * 55)
print("           MODEL PERFORMANCE BREAKDOWN BY LANGUAGE          ")
print("=" * 55)
print(lang_perf_df.to_string(index=False))
print("=" * 55)

In [31]:
from sklearn.metrics import classification_report

print("DistilBERT metrics by sentiment:")
print(
    classification_report(
        true_labels,
        preds,
        labels=[0, 1, 2],
        target_names=["negative", "neutral", "positive"],
        zero_division=0,
        digits=4,
    )
)